# Fama-French Size Factor (SMB) 성과 분석

한국 주식시장 사이즈 팩터(Small Minus Big) 성과를 시각화하고 해석합니다.
- 데이터: `quant_db` → `out/size_factor_monthly.csv`
- 기간: 2005-07 ~ 2026-03

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.figsize"] = (14, 5)

monthly = pd.read_csv("../out/size_factor_monthly.csv", index_col=0, parse_dates=True)
cumul = pd.read_csv("../out/size_factor_cumulative.csv", index_col=0, parse_dates=True)
print(f"기간: {monthly.index[0]:%Y-%m} ~ {monthly.index[-1]:%Y-%m} ({len(monthly)}개월)")
monthly.head()

## 1. 누적 수익률

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Small vs Big
ax = axes[0]
ax.plot(cumul.index, cumul["Small"], label="Small", linewidth=1.5)
ax.plot(cumul.index, cumul["Big"], label="Big", linewidth=1.5)
ax.set_title("Small vs Big 누적 수익률")
ax.legend()
ax.set_ylabel("Growth of ₩1")
ax.grid(alpha=0.3)

# SMB
ax = axes[1]
ax.plot(cumul.index, cumul["SMB"], color="tab:green", linewidth=1.5)
ax.axhline(1, color="gray", linestyle="--", linewidth=0.8)
ax.set_title("SMB 누적 수익률")
ax.set_ylabel("Growth of ₩1")
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 2. 성과 요약 통계

In [ ]:
def calc_stats(s):
    n = len(s)
    ann_ret = (1 + s).prod() ** (12 / n) - 1
    ann_vol = s.std() * np.sqrt(12)
    sharpe = ann_ret / ann_vol if ann_vol > 0 else 0
    cum = (1 + s).prod() - 1
    dd = (1 + s).cumprod() / (1 + s).cumprod().cummax() - 1
    t = s.mean() / (s.std() / np.sqrt(n)) if s.std() > 0 else 0
    return {
        "연환산 수익률": f"{ann_ret:.2%}",
        "연환산 변동성": f"{ann_vol:.2%}",
        "Sharpe Ratio": f"{sharpe:.2f}",
        "누적 수익률": f"{cum:.2%}",
        "Max Drawdown": f"{dd.min():.2%}",
        "승률 (월)": f"{(s > 0).mean():.1%}",
        "t-stat": f"{t:.2f}",
        "개월수": n,
    }

stats = pd.DataFrame({c: calc_stats(monthly[c].dropna()) for c in ["Small", "Big", "SMB"]})
stats

## 3. 연도별 수익률 히트맵

In [ ]:
yearly = monthly.groupby(monthly.index.year).apply(lambda x: (1 + x).prod() - 1)
yearly.index.name = "Year"

fig, ax = plt.subplots(figsize=(8, 12))
im = ax.imshow(yearly.values, cmap="RdYlGn", aspect="auto", vmin=-0.5, vmax=0.5)
ax.set_yticks(range(len(yearly)))
ax.set_yticklabels(yearly.index)
ax.set_xticks(range(3))
ax.set_xticklabels(["Big", "Small", "SMB"])

for i in range(len(yearly)):
    for j in range(3):
        val = yearly.iloc[i, j]
        ax.text(j, i, f"{val:.1%}", ha="center", va="center",
                fontsize=9, color="black" if abs(val) < 0.3 else "white")

ax.set_title("연도별 수익률")
fig.colorbar(im, ax=ax, shrink=0.5, label="Return")
plt.tight_layout()
plt.show()

## 4. 롤링 성과 (36개월)

In [ ]:
window = 36
rolling_ret = monthly["SMB"].rolling(window).apply(lambda x: (1 + x).prod() ** (12 / window) - 1)
rolling_vol = monthly["SMB"].rolling(window).std() * np.sqrt(12)

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

ax = axes[0]
ax.plot(rolling_ret.index, rolling_ret, color="tab:blue", linewidth=1.2)
ax.axhline(0, color="gray", linestyle="--", linewidth=0.8)
ax.fill_between(rolling_ret.index, rolling_ret, 0,
                where=rolling_ret > 0, alpha=0.3, color="tab:blue", label="소형주 우위")
ax.fill_between(rolling_ret.index, rolling_ret, 0,
                where=rolling_ret < 0, alpha=0.3, color="tab:red", label="대형주 우위")
ax.set_title("SMB 36개월 롤링 연환산 수익률")
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.legend()
ax.grid(alpha=0.3)

ax = axes[1]
ax.plot(rolling_vol.index, rolling_vol, color="tab:orange", linewidth=1.2)
ax.set_title("SMB 36개월 롤링 변동성")
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Drawdown 분석

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)

for ax, col, color in zip(axes, ["Small", "Big", "SMB"],
                           ["tab:blue", "tab:red", "tab:green"]):
    cum = (1 + monthly[col]).cumprod()
    dd = cum / cum.cummax() - 1
    ax.fill_between(dd.index, dd, 0, color=color, alpha=0.4)
    ax.set_title(f"{col} Drawdown")
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
    ax.set_ylim(-0.7, 0)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 6. 기간별 비교

In [ ]:
periods = {
    "전체": monthly,
    "~2009 (GFC 포함)": monthly[:"2009"],
    "2010~2014": monthly["2010":"2014"],
    "2015~2019": monthly["2015":"2019"],
    "2020~ (코로나 이후)": monthly["2020":],
}

rows = []
for label, subset in periods.items():
    s = subset["SMB"].dropna()
    if len(s) == 0:
        continue
    ann = (1 + s).prod() ** (12 / len(s)) - 1
    vol = s.std() * np.sqrt(12)
    t = s.mean() / (s.std() / np.sqrt(len(s))) if s.std() > 0 else 0
    rows.append({
        "기간": label,
        "연환산 수익률": f"{ann:+.2%}",
        "변동성": f"{vol:.2%}",
        "t-stat": f"{t:.2f}",
        "유의(5%)": "O" if abs(t) > 1.96 else "X",
        "개월": len(s),
    })

pd.DataFrame(rows).set_index("기간")

## 7. 월간 수익률 분포

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

for ax, col, color in zip(axes, ["Small", "Big", "SMB"],
                           ["tab:blue", "tab:red", "tab:green"]):
    data = monthly[col].dropna()
    ax.hist(data, bins=40, color=color, alpha=0.7, edgecolor="white")
    ax.axvline(data.mean(), color="black", linestyle="--", linewidth=1,
               label=f"mean={data.mean():.3f}")
    ax.axvline(0, color="gray", linestyle="-", linewidth=0.8)
    ax.set_title(f"{col} 월간 수익률 분포")
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

## 해석 요약

| 항목 | 결과 |
|------|------|
| **전체 기간 SMB** | 연 -0.23%, t-stat 0.29 → 통계적으로 유의하지 않음 |
| **~2014** | 소형주 프리미엄 존재 (연 +1.3%), 다만 유의하지 않음 |
| **2020 이후** | 대형주 강세 (SMB 연 -9.5%), 반도체·2차전지 대형주 쏠림 |
| **2025** | 소형주 +101% 급등, SMB 반등 조짐이나 단기 현상 가능성 |

### 핵심 시사점

1. **한국 시장에서 사이즈 프리미엄은 구조적이지 않다** — 미국과 달리 일관된 소형주 프리미엄이 관찰되지 않음
2. **레짐 의존적** — 시장 사이클에 따라 소형주/대형주 우위가 번갈아 나타남
3. **단독 팩터로서의 활용 가치 제한적** — 다른 팩터(밸류, 모멘텀 등)와 결합 필요
4. **대형주 쏠림 가속화** — 2020년 이후 특정 대형주 집중도 심화로 SMB 악화